In [2]:
%pip install -U ccxt pandas numpy statsmodels tqdm

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.6 MB 11.3 MB/s eta 0:00:01
   -------------- ------------------------- 4.5/12.6 MB 11.7 MB/s eta 0:00:01
   ---------------------- ----------------- 7.1/12.6 MB 11.7 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.6 MB 11.7 MB/s eta 0:00:01
   -------------------------------------- - 12.1/12.6 MB 11.7 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 11.2 MB/s  0:00:01
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   -------- ------------------------------- 2.4/11.3 MB 11.7 MB/s eta 0:00:01
   -------------- ------------------------- 4.2/11.3 MB 11.7 MB/s eta 0:00:01
   -------------------- ------------------- 5.8/11.3 MB 9.1 MB/s eta 0:00:01
   ----------------------------- ---------- 8.4/11.3 MB 9.9 MB/s eta 0:00:01
   --------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.65.1 requires numpy<2.5,>=1.22, but you have numpy 2.5.3 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\crp\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [9]:

# ============================================================
# 0. ИМПОРТЫ
# ============================================================

import asyncio
import os
import time
import warnings
from datetime import datetime, timezone

import ccxt.async_support as ccxt
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


# ============================================================
# 1. НАСТРОЙКИ
# ============================================================

# ------------------------------------------------------------
# Биржи.
#
# Примеры:
# EXCHANGES = ["mexc", "gate"]
# EXCHANGES = ["binance", "bybit"]
# EXCHANGES = ["okx", "bybit"]
# ------------------------------------------------------------

EXCHANGES = ["mexc", "gate"]

# Только USDT-рынки.
QUOTE = "USDT"

# ------------------------------------------------------------
# Какие типы ног разрешены.
# ------------------------------------------------------------

LEG_TYPE_PAIRS = [
    ("spot", "swap"),
]

# True = только Exchange A ↔ Exchange B.
# Не сравниваем spot/perp внутри одной и той же биржи.
CROSS_EXCHANGE_ONLY = True

# True = берём только USDT-linear perpetuals.
# Inverse-контракты не берём.
ONLY_LINEAR_SWAPS = True

# ------------------------------------------------------------
# Historical окна.
# ------------------------------------------------------------

FAST_TIMEFRAME = "5m"
FAST_DAYS = 2

SLOW_TIMEFRAME = "15m"
SLOW_DAYS = 7

# 0 = сканировать все общие монеты.
#
# Для теста можно поставить 100.
# Для полного прохода оставь 0.
MAX_BASES_TO_SCAN = 0

# Одновременные запросы.
# Для MEXC/Gate лучше 3-4, а не 20.
MAX_CONCURRENT_REQUESTS = 4

# ------------------------------------------------------------
# УСЛОВИЯ ИСТОРИЧЕСКОГО SPREAD.
# ------------------------------------------------------------

# ADF p-value ≤ 0.05 = spread считаем стационарным.
ADF_PVALUE_MAX = 0.05

# Минимум общих синхронизированных свечей.
MIN_FAST_BARS = 400
MIN_SLOW_BARS = 400

# Устойчивый размах spread:
# q95 - q05.
#
# 0.04 ≈ 4%.
MIN_Q95_Q05_RANGE = 0.02
MAX_Q95_Q05_RANGE = 0.35

# Минимальная стандартная девиация spread.
MIN_SPREAD_STD = 0.006

# Half-life в барах.
#
# Для 5m:
# 288 bars = 24 часа.
#
# Для 15m:
# 288 bars = 72 часа.
MIN_HALFLIFE_BARS = 2
MAX_FAST_HALFLIFE_BARS = 288
MAX_SLOW_HALFLIFE_BARS = 288

# Частота mean-reversion.
MIN_ZERO_CROSSINGS_PER_DAY = 3.0
MIN_ONE_SIGMA_ENTRIES_PER_DAY = 2.0

# ------------------------------------------------------------
# УСЛОВИЯ РЕАЛЬНОГО ИСПОЛНЕНИЯ.
# ------------------------------------------------------------

# На какую сумму проверяем стакан каждой ноги.
TEST_NOTIONAL_USDT = 100.0

# Запрашиваемая глубина order book.
ORDERBOOK_LIMIT = 100

# Максимально допустимое проскальзывание одной ноги.
# 0.0015 = 0.15%.
MAX_LEG_SLIPPAGE = 0.0045

# Резерв taker fee одной сделки на одной ноге.
DEFAULT_TAKER_FEE = 0.001

# Грубая оценка числа funding settlement в день.
FUNDING_EVENTS_PER_DAY = 3

# 1σ движение должно быть больше costs в N раз.
MIN_COST_COVERAGE_MULTIPLIER = 1.50

# ------------------------------------------------------------
# КУДА СОХРАНЯТЬ CSV.
# ------------------------------------------------------------

OUTPUT_DIR = "stat_arb_scan_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUEST_SEMAPHORE = asyncio.Semaphore(
    MAX_CONCURRENT_REQUESTS
)

print("=" * 85)
print("STAT-ARB SCANNER: настройки загружены")
print("=" * 85)
print(f"Биржи: {EXCHANGES}")
print(f"Типы связок: {LEG_TYPE_PAIRS}")
print(f"Fast окно: {FAST_DAYS}d / {FAST_TIMEFRAME}")
print(f"Slow окно: {SLOW_DAYS}d / {SLOW_TIMEFRAME}")
print(f"Минимальный q95-q05 range: {MIN_Q95_Q05_RANGE * 100:.2f}%")
print(f"Тестовый размер стакана: ${TEST_NOTIONAL_USDT:,.0f}")
print()




STAT-ARB SCANNER: настройки загружены
Биржи: ['mexc', 'gate']
Типы связок: [('spot', 'swap')]
Fast окно: 2d / 5m
Slow окно: 7d / 15m
Минимальный q95-q05 range: 2.00%
Тестовый размер стакана: $100



In [10]:
# ============================================================
# CROSS-EXCHANGE STAT-ARB SCANNER
#
# Что делает:
# 1. Подключается к двум биржам.
# 2. Берёт все общие USDT Spot и USDT-linear Perpetual рынки.
# 3. Сравнивает один и тот же asset между биржами:
#    - Spot ↔ Spot
#    - Futures ↔ Futures
#    - Spot ↔ Futures
#    - Futures ↔ Spot
# 4. Быстрый отбор: 2 дня / 5 минут.
# 5. Подтверждение: 7 дней / 15 минут.
# 6. Расчёт:
#    beta, ADF, q95-q05 range, std, half-life,
#    zero-crossings, ±1σ / ±2σ entries.
# 7. Проверка стакана:
#    VWAP, slippage, executable edge, fees, funding.
# 8. Сохраняет CSV со всеми этапами.
#
# ВАЖНО:
# Перед запуском должны работать:
# import numpy, scipy, pandas, statsmodels, ccxt
# ============================================================

# ============================================================
# 2. БАЗОВЫЕ ФУНКЦИИ
# ============================================================

def now_ms():
    return int(time.time() * 1000)


def timeframe_to_ms(timeframe):
    multipliers = {
        "m": 60_000,
        "h": 3_600_000,
        "d": 86_400_000,
    }

    return int(timeframe[:-1]) * multipliers[timeframe[-1]]


def timeframe_to_minutes(timeframe):
    return timeframe_to_ms(timeframe) / 60_000


def safe_float(value, default=np.nan):
    try:
        value = float(value)

        if np.isfinite(value):
            return value

        return default

    except Exception:
        return default


def pct(value, digits=3):
    if pd.isna(value):
        return "нет данных"

    return f"{value:.{digits}f}%"


def market_kind(market):
    """
    Возвращает:
    spot
    swap
    None
    """
    if market.get("spot", False):
        return "spot"

    if market.get("swap", False):
        if ONLY_LINEAR_SWAPS and not market.get("linear", False):
            return None

        return "swap"

    return None


async def create_exchange(exchange_id):
    exchange_class = getattr(ccxt, exchange_id)

    exchange = exchange_class({
        "enableRateLimit": True,
        "timeout": 30_000,
        "options": {
            "adjustForTimeDifference": True,
        },
    })

    await exchange.load_markets()

    return exchange


def leg_key(leg):
    return (
        leg["exchange_id"],
        leg["kind"],
        leg["symbol"],
    )


def build_universe(exchange):
    """
    Формат результата:

    {
        ("BTC", "spot"): {...},
        ("BTC", "swap"): {...},
        ("ETH", "spot"): {...},
        ...
    }
    """
    universe = {}

    for symbol, market in exchange.markets.items():
        try:
            if not market.get("active", True):
                continue

            if market.get("quote") != QUOTE:
                continue

            kind = market_kind(market)

            if kind is None:
                continue

            base = market.get("base")

            if not base:
                continue

            key = (base, kind)

            if key not in universe:
                universe[key] = {
                    "exchange_id": exchange.id,
                    "symbol": symbol,
                    "base": base,
                    "quote": market.get("quote"),
                    "kind": kind,
                    "market": market,
                }

        except Exception:
            continue

    return universe


# ============================================================
# 3. ЗАГРУЗКА OHLCV, СТАКАНА И FUNDING
# ============================================================

async def fetch_ohlcv_history(exchange, symbol, timeframe, days):
    """
    Загружает OHLCV с пагинацией.

    Для заданных окон обычно достаточно одного limit=1000:
    - 2d / 5m = ~576 баров;
    - 7d / 15m = ~672 бара.
    """
    since = now_ms() - int(days * 86_400_000)
    until = now_ms()

    tf_ms = timeframe_to_ms(timeframe)
    limit = 1000

    all_rows = []

    while since < until:
        try:
            async with REQUEST_SEMAPHORE:
                rows = await exchange.fetch_ohlcv(
                    symbol=symbol,
                    timeframe=timeframe,
                    since=since,
                    limit=limit,
                )

        except Exception:
            break

        if not rows:
            break

        all_rows.extend(rows)

        last_timestamp = rows[-1][0]
        next_since = last_timestamp + tf_ms

        if next_since <= since:
            break

        since = next_since

        if len(rows) < limit:
            break

    if not all_rows:
        return pd.DataFrame(
            columns=["close", "volume"]
        )

    df = pd.DataFrame(
        all_rows,
        columns=[
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
        ],
    )

    df = df.drop_duplicates(
        subset=["timestamp"]
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        unit="ms",
        utc=True,
    )

    df = (
        df
        .set_index("timestamp")
        .sort_index()
    )

    df["close"] = pd.to_numeric(
        df["close"],
        errors="coerce",
    )

    df["volume"] = pd.to_numeric(
        df["volume"],
        errors="coerce",
    )

    df = df[df["close"] > 0]

    return df[["close", "volume"]]


async def fetch_orderbook_safe(exchange, symbol):
    try:
        async with REQUEST_SEMAPHORE:
            book = await exchange.fetch_order_book(
                symbol=symbol,
                limit=ORDERBOOK_LIMIT,
            )

        return book

    except Exception:
        return None


async def fetch_funding_safe(exchange, symbol, kind):
    """
    Spot → funding = 0.
    Swap → пробуем получить текущую funding rate.
    """
    if kind != "swap":
        return {
            "available": True,
            "funding_rate": 0.0,
        }

    try:
        if not exchange.has.get(
            "fetchFundingRate",
            False,
        ):
            return {
                "available": False,
                "funding_rate": np.nan,
            }

        async with REQUEST_SEMAPHORE:
            data = await exchange.fetch_funding_rate(
                symbol
            )

        rate = data.get("fundingRate")

        if rate is None:
            rate = data.get(
                "previousFundingRate"
            )

        rate = safe_float(rate)

        return {
            "available": np.isfinite(rate),
            "funding_rate": rate,
        }

    except Exception:
        return {
            "available": False,
            "funding_rate": np.nan,
        }


async def load_many_ohlcv(
    exchanges,
    legs,
    timeframe,
    days,
    progress_name,
):
    """
    Загружает OHLCV для уникальных ног.

    Одна и та же нога может участвовать в:
    - Spot-Spot;
    - Spot-Swap;
    - Swap-Swap.

    Поэтому качаем её один раз и используем из cache.
    """
    unique_legs = {}

    for leg in legs:
        unique_legs[leg_key(leg)] = leg

    async def load_one(key, leg):
        try:
            exchange = exchanges[leg["exchange_id"]]

            df = await fetch_ohlcv_history(
                exchange=exchange,
                symbol=leg["symbol"],
                timeframe=timeframe,
                days=days,
            )

            return key, df, ""

        except Exception as e:
            return (
                key,
                pd.DataFrame(
                    columns=["close", "volume"]
                ),
                f"{type(e).__name__}: {e}",
            )

    tasks = [
        asyncio.create_task(
            load_one(key, leg)
        )
        for key, leg in unique_legs.items()
    ]

    cache = {}
    errors = {}

    for future in tqdm(
        asyncio.as_completed(tasks),
        total=len(tasks),
        desc=progress_name,
    ):
        key, df, error = await future

        cache[key] = df

        if error:
            errors[key] = error

    return cache, errors


# ============================================================
# 4. SPREAD MATH
# ============================================================

def estimate_beta_and_spread(
    price_a,
    price_b,
):
    """
    Регрессия:

    log(P_A) = alpha + beta * log(P_B) + residual

    residual = stationary spread candidate.
    beta = hedge ratio.
    """
    x = np.log(
        price_b.astype(float).values
    )

    y = np.log(
        price_a.astype(float).values
    )

    X = np.column_stack([
        np.ones(len(x)),
        x,
    ])

    alpha, beta = np.linalg.lstsq(
        X,
        y,
        rcond=None,
    )[0]

    spread = y - (
        alpha + beta * x
    )

    return (
        float(alpha),
        float(beta),
        spread,
    )


def estimate_half_life(spread):
    """
    ΔS_t = c + gamma*S_(t-1) + epsilon

    half-life = -ln(2) / gamma
    только при gamma < 0.

    Результат в барах.
    """
    spread = (
        pd.Series(spread)
        .dropna()
        .values
    )

    if len(spread) < 30:
        return np.inf

    lagged = spread[:-1]
    delta = np.diff(spread)

    X = np.column_stack([
        np.ones(len(lagged)),
        lagged,
    ])

    try:
        _, gamma = np.linalg.lstsq(
            X,
            delta,
            rcond=None,
        )[0]

    except Exception:
        return np.inf

    if not np.isfinite(gamma):
        return np.inf

    if gamma >= 0:
        return np.inf

    half_life = -np.log(2) / gamma

    if not np.isfinite(half_life):
        return np.inf

    if half_life <= 0:
        return np.inf

    return float(half_life)


def count_level_crossings(
    values,
    level=0.0,
):
    """
    Сколько раз ряд пересёк заданный level.

    Для z-score и level=0:
    количество возвратов через среднее.
    """
    x = (
        pd.Series(values)
        .dropna()
        .values
        - level
    )

    if len(x) < 2:
        return 0

    signs = np.sign(x).astype(float)

    signs[signs == 0] = np.nan

    signs = (
        pd.Series(signs)
        .ffill()
        .bfill()
        .values
    )

    return int(
        np.sum(
            signs[1:] * signs[:-1] < 0
        )
    )


def count_sigma_entries(
    zscore,
    sigma=1.0,
):
    """
    Считает независимые входы:
    - в z >= +sigma;
    - в z <= -sigma.

    Это число событий, не число свечей.
    """
    z = (
        pd.Series(zscore)
        .dropna()
        .values
    )

    if len(z) < 2:
        return 0

    upper = z >= sigma
    lower = z <= -sigma

    entered_upper = np.sum(
        (~upper[:-1]) & upper[1:]
    )

    entered_lower = np.sum(
        (~lower[:-1]) & lower[1:]
    )

    return int(
        entered_upper + entered_lower
    )


def analyse_pair_ohlcv(
    df_a,
    df_b,
    timeframe,
):
    """
    Historical analysis одной связки.
    """
    aligned = (
        df_a[["close"]]
        .rename(
            columns={"close": "price_a"}
        )
        .join(
            df_b[["close"]].rename(
                columns={"close": "price_b"}
            ),
            how="inner",
        )
        .dropna()
    )

    aligned = aligned[
        (aligned["price_a"] > 0)
        & (aligned["price_b"] > 0)
    ]

    min_bars = (
        MIN_FAST_BARS
        if timeframe == FAST_TIMEFRAME
        else MIN_SLOW_BARS
    )

    if len(aligned) < min_bars:
        return {
            "ok": False,
            "bars": len(aligned),
            "reason": (
                f"Недостаточно общих свечей: "
                f"{len(aligned)} < {min_bars}"
            ),
        }

    alpha, beta, spread_array = (
        estimate_beta_and_spread(
            aligned["price_a"],
            aligned["price_b"],
        )
    )

    spread = pd.Series(
        spread_array,
        index=aligned.index,
        name="spread",
    )

    spread_mean = spread.mean()
    spread_std = spread.std(ddof=1)

    if not np.isfinite(spread_std):
        return {
            "ok": False,
            "bars": len(aligned),
            "reason": "Spread std невалиден",
        }

    if spread_std <= 0:
        return {
            "ok": False,
            "bars": len(aligned),
            "reason": "Spread std равен нулю",
        }

    zscore = (
        spread - spread_mean
    ) / spread_std

    try:
        adf_pvalue = float(
            adfuller(
                spread.values,
                autolag="AIC",
            )[1]
        )

    except Exception:
        adf_pvalue = np.nan

    q05 = float(
        spread.quantile(0.05)
    )

    q50 = float(
        spread.quantile(0.50)
    )

    q95 = float(
        spread.quantile(0.95)
    )

    q95_q05_range = q95 - q05

    half_life_bars = estimate_half_life(
        spread
    )

    zero_crossings = count_level_crossings(
        zscore,
        level=0.0,
    )

    one_sigma_entries = count_sigma_entries(
        zscore,
        sigma=1.0,
    )

    two_sigma_entries = count_sigma_entries(
        zscore,
        sigma=2.0,
    )

    minutes_per_bar = timeframe_to_minutes(
        timeframe
    )

    period_days = (
        len(aligned)
        * minutes_per_bar
        / 1_440
    )

    zero_crossings_per_day = (
        zero_crossings / period_days
        if period_days > 0
        else 0.0
    )

    one_sigma_entries_per_day = (
        one_sigma_entries / period_days
        if period_days > 0
        else 0.0
    )

    two_sigma_entries_per_day = (
        two_sigma_entries / period_days
        if period_days > 0
        else 0.0
    )

    max_half_life = (
        MAX_FAST_HALFLIFE_BARS
        if timeframe == FAST_TIMEFRAME
        else MAX_SLOW_HALFLIFE_BARS
    )

    stationary_ok = (
        np.isfinite(adf_pvalue)
        and adf_pvalue <= ADF_PVALUE_MAX
    )

    range_ok = (
        MIN_Q95_Q05_RANGE
        <= q95_q05_range
        <= MAX_Q95_Q05_RANGE
    )

    std_ok = (
        spread_std >= MIN_SPREAD_STD
    )

    half_life_ok = (
        MIN_HALFLIFE_BARS
        <= half_life_bars
        <= max_half_life
    )

    crossings_ok = (
        zero_crossings_per_day
        >= MIN_ZERO_CROSSINGS_PER_DAY
    )

    activity_ok = (
        one_sigma_entries_per_day
        >= MIN_ONE_SIGMA_ENTRIES_PER_DAY
    )

    historical_ok = all([
        stationary_ok,
        range_ok,
        std_ok,
        half_life_ok,
        crossings_ok,
        activity_ok,
    ])

    return {
        "ok": True,
        "bars": len(aligned),
        "reason": "",

        "alpha": alpha,
        "beta": beta,

        "adf_pvalue": adf_pvalue,

        "spread_mean": float(spread_mean),
        "spread_std": float(spread_std),

        "spread_q05": q05,
        "spread_q50": q50,
        "spread_q95": q95,

        "q95_q05_range": float(
            q95_q05_range
        ),

        "half_life_bars": float(
            half_life_bars
        ),

        "zero_crossings": zero_crossings,

        "zero_crossings_per_day": float(
            zero_crossings_per_day
        ),

        "one_sigma_entries": one_sigma_entries,

        "one_sigma_entries_per_day": float(
            one_sigma_entries_per_day
        ),

        "two_sigma_entries": two_sigma_entries,

        "two_sigma_entries_per_day": float(
            two_sigma_entries_per_day
        ),

        "latest_spread": float(
            spread.iloc[-1]
        ),

        "latest_zscore": float(
            zscore.iloc[-1]
        ),

        "stationary_ok": stationary_ok,
        "range_ok": range_ok,
        "std_ok": std_ok,
        "half_life_ok": half_life_ok,
        "crossings_ok": crossings_ok,
        "activity_ok": activity_ok,

        "historical_ok": historical_ok,
    }


def historical_rejection_reason(metrics):
    """
    Текстовое объяснение historical reject.
    """
    if not metrics.get("ok", False):
        return metrics.get(
            "reason",
            "Historical analysis error",
        )

    reasons = []

    if not metrics["stationary_ok"]:
        reasons.append(
            f"ADF={metrics['adf_pvalue']:.4f} "
            f"> {ADF_PVALUE_MAX}"
        )

    if not metrics["range_ok"]:
        reasons.append(
            f"range={metrics['q95_q05_range'] * 100:.2f}% "
            f"вне {MIN_Q95_Q05_RANGE * 100:.2f}%"
            f"–{MAX_Q95_Q05_RANGE * 100:.2f}%"
        )

    if not metrics["std_ok"]:
        reasons.append(
            f"std={metrics['spread_std'] * 100:.3f}% "
            f"< {MIN_SPREAD_STD * 100:.3f}%"
        )

    if not metrics["half_life_ok"]:
        reasons.append(
            f"half-life="
            f"{metrics['half_life_bars']:.1f} bars"
        )

    if not metrics["crossings_ok"]:
        reasons.append(
            f"zero-crossings/day="
            f"{metrics['zero_crossings_per_day']:.2f} "
            f"< {MIN_ZERO_CROSSINGS_PER_DAY:.2f}"
        )

    if not metrics["activity_ok"]:
        reasons.append(
            f"±1σ entries/day="
            f"{metrics['one_sigma_entries_per_day']:.2f} "
            f"< {MIN_ONE_SIGMA_ENTRIES_PER_DAY:.2f}"
        )

    if not reasons:
        return ""

    return " | ".join(reasons)


# ============================================================
# 5. VWAP И СТАКАН
# ============================================================

def vwap_from_book(
    levels,
    quote_notional,
):
    """
    Расчёт VWAP для заданного объёма USDT.

    ВАЖНО:
    Поддерживает форматы уровня стакана:
    [price, amount]
    [price, amount, timestamp]
    [price, amount, ...].

    Поэтому берём level[0] и level[1],
    а все дополнительные поля игнорируем.
    """
    if not levels:
        return {
            "filled": False,
            "vwap": np.nan,
            "quote_filled": 0.0,
            "base_filled": 0.0,
        }

    remaining_quote = float(
        quote_notional
    )

    total_quote = 0.0
    total_base = 0.0

    for level in levels:
        if level is None:
            continue

        if len(level) < 2:
            continue

        price = safe_float(level[0])
        amount = safe_float(level[1])

        if not np.isfinite(price):
            continue

        if not np.isfinite(amount):
            continue

        if price <= 0 or amount <= 0:
            continue

        level_capacity_quote = (
            price * amount
        )

        take_quote = min(
            remaining_quote,
            level_capacity_quote,
        )

        take_base = take_quote / price

        total_quote += take_quote
        total_base += take_base

        remaining_quote -= take_quote

        if remaining_quote <= max(
            1e-8,
            quote_notional * 1e-6,
        ):
            break

    filled = remaining_quote <= max(
        1e-8,
        quote_notional * 1e-6,
    )

    vwap = (
        total_quote / total_base
        if total_base > 0
        else np.nan
    )

    return {
        "filled": filled,
        "vwap": vwap,
        "quote_filled": total_quote,
        "base_filled": total_base,
    }


def analyse_orderbook(
    orderbook,
    notional_usdt,
):
    """
    Проверяет одну ногу:

    - best bid / ask;
    - можно ли купить на notional;
    - можно ли продать на notional;
    - buy VWAP;
    - sell VWAP;
    - slippage.
    """
    if orderbook is None:
        return {
            "ok": False,
            "reason": "Стакан не получен",
        }

    bids = orderbook.get("bids") or []
    asks = orderbook.get("asks") or []

    if not bids or not asks:
        return {
            "ok": False,
            "reason": "Пустой стакан",
        }

    if len(bids[0]) < 2:
        return {
            "ok": False,
            "reason": "Некорректный best bid",
        }

    if len(asks[0]) < 2:
        return {
            "ok": False,
            "reason": "Некорректный best ask",
        }

    best_bid = safe_float(
        bids[0][0]
    )

    best_ask = safe_float(
        asks[0][0]
    )

    if not np.isfinite(best_bid):
        return {
            "ok": False,
            "reason": "Невалидный best bid",
        }

    if not np.isfinite(best_ask):
        return {
            "ok": False,
            "reason": "Невалидный best ask",
        }

    if best_bid <= 0 or best_ask <= 0:
        return {
            "ok": False,
            "reason": "Нулевая цена в стакане",
        }

    buy = vwap_from_book(
        levels=asks,
        quote_notional=notional_usdt,
    )

    sell = vwap_from_book(
        levels=bids,
        quote_notional=notional_usdt,
    )

    if not buy["filled"]:
        return {
            "ok": False,
            "reason": (
                f"Недостаточная ask-глубина "
                f"для ${notional_usdt:,.0f}"
            ),
        }

    if not sell["filled"]:
        return {
            "ok": False,
            "reason": (
                f"Недостаточная bid-глубина "
                f"для ${notional_usdt:,.0f}"
            ),
        }

    buy_slippage = (
        buy["vwap"] - best_ask
    ) / best_ask

    sell_slippage = (
        best_bid - sell["vwap"]
    ) / best_bid

    return {
        "ok": True,
        "reason": "",

        "best_bid": best_bid,
        "best_ask": best_ask,

        "mid": (
            best_bid + best_ask
        ) / 2,

        "buy_vwap": buy["vwap"],
        "sell_vwap": sell["vwap"],

        "buy_slippage": buy_slippage,
        "sell_slippage": sell_slippage,

        "max_slippage": max(
            buy_slippage,
            sell_slippage,
        ),
    }


def executable_edges(
    book_a,
    book_b,
):
    """
    Реальные current edge по VWAP.

    short A / long B:
    - продаём A по sell VWAP;
    - покупаем B по buy VWAP.

    long A / short B:
    - покупаем A по buy VWAP;
    - продаём B по sell VWAP.
    """
    edge_short_a_long_b = (
        book_a["sell_vwap"]
        / book_b["buy_vwap"]
        - 1
    )

    edge_long_a_short_b = (
        book_b["sell_vwap"]
        / book_a["buy_vwap"]
        - 1
    )

    return {
        "edge_short_a_long_b": (
            edge_short_a_long_b
        ),

        "edge_long_a_short_b": (
            edge_long_a_short_b
        ),

        "best_current_edge": max(
            edge_short_a_long_b,
            edge_long_a_short_b,
        ),
    }


# ============================================================
# 6. ФОРМИРОВАНИЕ КАНДИДАТОВ
# ============================================================

def build_candidates(universes):
    """
    Формирует все возможные сравнения
    для одного и того же base asset.
    """
    exchange_ids = list(universes.keys())

    all_bases = set()

    for universe in universes.values():
        for base, kind in universe.keys():
            all_bases.add(base)

    all_bases = sorted(all_bases)

    if MAX_BASES_TO_SCAN > 0:
        all_bases = all_bases[
            :MAX_BASES_TO_SCAN
        ]

    candidates = []
    seen = set()

    for base in all_bases:
        for i, exchange_a_id in enumerate(
            exchange_ids
        ):
            for exchange_b_id in exchange_ids[
                i + 1:
            ]:
                for kind_a, kind_b in LEG_TYPE_PAIRS:
                    leg_a = universes[
                        exchange_a_id
                    ].get(
                        (base, kind_a)
                    )

                    leg_b = universes[
                        exchange_b_id
                    ].get(
                        (base, kind_b)
                    )

                    if leg_a is None:
                        continue

                    if leg_b is None:
                        continue

                    if (
                        CROSS_EXCHANGE_ONLY
                        and exchange_a_id
                        == exchange_b_id
                    ):
                        continue

                    pair_id = (
                        f"{base}|"
                        f"{exchange_a_id}:"
                        f"{kind_a}:"
                        f"{leg_a['symbol']}|"
                        f"{exchange_b_id}:"
                        f"{kind_b}:"
                        f"{leg_b['symbol']}"
                    )

                    if pair_id in seen:
                        continue

                    seen.add(pair_id)

                    candidates.append({
                        "pair_id": pair_id,
                        "base": base,
                        "leg_a": leg_a,
                        "leg_b": leg_b,
                    })

    return candidates


def candidate_row(candidate):
    """
    Базовые поля одной связки для DataFrame.
    """
    leg_a = candidate["leg_a"]
    leg_b = candidate["leg_b"]

    return {
        "pair_id": candidate["pair_id"],
        "base": candidate["base"],

        "exchange_a": leg_a["exchange_id"],
        "market_a": leg_a["kind"],
        "symbol_a": leg_a["symbol"],

        "exchange_b": leg_b["exchange_id"],
        "market_b": leg_b["kind"],
        "symbol_b": leg_b["symbol"],
    }


def pair_label(row):
    return (
        f"{row['base']} | "
        f"{row['exchange_a'].upper()} "
        f"{row['market_a'].upper()} "
        f"({row['symbol_a']}) ↔ "
        f"{row['exchange_b'].upper()} "
        f"{row['market_b'].upper()} "
        f"({row['symbol_b']})"
    )


def metrics_to_row(prefix, metrics):
    """
    Historical metrics -> плоские CSV/DataFrame колонки.
    """
    if not metrics.get("ok", False):
        return {
            f"{prefix}_analysis_ok": False,
            f"{prefix}_reason": metrics.get(
                "reason",
                "",
            ),
            f"{prefix}_bars": metrics.get(
                "bars",
                0,
            ),
        }

    return {
        f"{prefix}_analysis_ok": True,
        f"{prefix}_reason": "",
        f"{prefix}_bars": metrics["bars"],

        f"{prefix}_alpha": metrics["alpha"],
        f"{prefix}_beta": metrics["beta"],

        f"{prefix}_adf_pvalue": (
            metrics["adf_pvalue"]
        ),

        f"{prefix}_spread_std_pct": (
            metrics["spread_std"] * 100
        ),

        f"{prefix}_spread_q05_pct": (
            (
                np.exp(
                    metrics["spread_q05"]
                ) - 1
            ) * 100
        ),

        f"{prefix}_spread_q50_pct": (
            (
                np.exp(
                    metrics["spread_q50"]
                ) - 1
            ) * 100
        ),

        f"{prefix}_spread_q95_pct": (
            (
                np.exp(
                    metrics["spread_q95"]
                ) - 1
            ) * 100
        ),

        f"{prefix}_q95_q05_range_pct": (
            metrics["q95_q05_range"] * 100
        ),

        f"{prefix}_half_life_bars": (
            metrics["half_life_bars"]
        ),

        f"{prefix}_zero_crossings": (
            metrics["zero_crossings"]
        ),

        f"{prefix}_zero_crossings_day": (
            metrics["zero_crossings_per_day"]
        ),

        f"{prefix}_one_sigma_entries": (
            metrics["one_sigma_entries"]
        ),

        f"{prefix}_one_sigma_entries_day": (
            metrics["one_sigma_entries_per_day"]
        ),

        f"{prefix}_two_sigma_entries": (
            metrics["two_sigma_entries"]
        ),

        f"{prefix}_two_sigma_entries_day": (
            metrics["two_sigma_entries_per_day"]
        ),

        f"{prefix}_latest_zscore": (
            metrics["latest_zscore"]
        ),

        f"{prefix}_latest_spread_pct": (
            (
                np.exp(
                    metrics["latest_spread"]
                ) - 1
            ) * 100
        ),

        f"{prefix}_historical_ok": (
            metrics["historical_ok"]
        ),

        f"{prefix}_historical_rejection_reason": (
            historical_rejection_reason(
                metrics
            )
        ),
    }


# ============================================================
# 7. ОСНОВНОЙ СКАНЕР
# ============================================================

async def run_full_scanner():
    started_at = datetime.now(
        timezone.utc
    )

    exchanges = {}

    try:
        # ----------------------------------------------------
        # STEP 1 — EXCHANGES
        # ----------------------------------------------------

        print("=" * 70)
        print("STEP 1/7 — Подключение к биржам")
        print("=" * 70)

        for exchange_id in EXCHANGES:
            try:
                exchange = await create_exchange(
                    exchange_id
                )

                exchanges[exchange_id] = exchange

                print(
                    f"OK: {exchange_id} | "
                    f"markets={len(exchange.markets)}"
                )

            except Exception as e:
                print(
                    f"FAIL: {exchange_id} | "
                    f"{type(e).__name__}: {e}"
                )

        if len(exchanges) < 2:
            raise RuntimeError(
                "Нужно минимум две доступные биржи."
            )

        # ----------------------------------------------------
        # STEP 2 — UNIVERSE
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(
            "STEP 2/7 — Spot / "
            "USDT-linear perpetual universe"
        )
        print("=" * 70)

        universes = {}

        for exchange_id, exchange in exchanges.items():
            universe = build_universe(exchange)

            universes[exchange_id] = universe

            spot_count = sum(
                1
                for base, kind in universe.keys()
                if kind == "spot"
            )

            swap_count = sum(
                1
                for base, kind in universe.keys()
                if kind == "swap"
            )

            print(
                f"{exchange_id}: "
                f"spot={spot_count}, "
                f"linear swaps={swap_count}"
            )

        candidates = build_candidates(
            universes
        )

        print()
        print(
            f"Всего связок: {len(candidates)}"
        )

        if not candidates:
            raise RuntimeError(
                "Не найдено общих связок."
            )

        # ----------------------------------------------------
        # STEP 3 — FAST OHLCV
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(
            f"STEP 3/7 — Загрузка "
            f"{FAST_DAYS}d / {FAST_TIMEFRAME}"
        )
        print("=" * 70)

        fast_legs = []

        for candidate in candidates:
            fast_legs.append(
                candidate["leg_a"]
            )

            fast_legs.append(
                candidate["leg_b"]
            )

        fast_cache, fast_download_errors = (
            await load_many_ohlcv(
                exchanges=exchanges,
                legs=fast_legs,
                timeframe=FAST_TIMEFRAME,
                days=FAST_DAYS,
                progress_name=(
                    f"OHLCV {FAST_TIMEFRAME}"
                ),
            )
        )

        print(
            f"Загружено unique ног: "
            f"{len(fast_cache)}"
        )

        print(
            f"Ошибок загрузки: "
            f"{len(fast_download_errors)}"
        )

        # ----------------------------------------------------
        # STEP 4 — FAST ANALYSIS
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(
            f"STEP 4/7 — Анализ "
            f"{FAST_DAYS}d / {FAST_TIMEFRAME}"
        )
        print("=" * 70)

        fast_rows = []
        fast_passed = []

        for candidate in tqdm(
            candidates,
            desc="FAST historical analysis",
        ):
            df_a = fast_cache.get(
                leg_key(candidate["leg_a"]),
                pd.DataFrame(
                    columns=["close", "volume"]
                ),
            )

            df_b = fast_cache.get(
                leg_key(candidate["leg_b"]),
                pd.DataFrame(
                    columns=["close", "volume"]
                ),
            )

            metrics = analyse_pair_ohlcv(
                df_a=df_a,
                df_b=df_b,
                timeframe=FAST_TIMEFRAME,
            )

            row = candidate_row(candidate)

            row.update(
                metrics_to_row(
                    "fast",
                    metrics,
                )
            )

            fast_rows.append(row)

            if metrics.get(
                "historical_ok",
                False,
            ):
                candidate["fast_metrics"] = metrics

                fast_passed.append(
                    candidate
                )

        fast_df = pd.DataFrame(fast_rows)

        print()
        print(
            f"Прошли fast historical filter: "
            f"{len(fast_passed)} / {len(candidates)}"
        )

        timestamp = started_at.strftime(
            "%Y%m%d_%H%M%S"
        )

        fast_path = os.path.join(
            OUTPUT_DIR,
            f"01_fast_all_{timestamp}.csv",
        )

        fast_df.to_csv(
            fast_path,
            index=False,
            encoding="utf-8-sig",
        )

        if not fast_passed:
            print()
            print(
                "Ни одна связка не прошла "
                "fast-фильтр."
            )

            return {
                "fast_df": fast_df,
                "slow_df": pd.DataFrame(),
                "tradeable_df": pd.DataFrame(),
                "execution_rejected_df": pd.DataFrame(),
            }

        # ----------------------------------------------------
        # STEP 5 — SLOW OHLCV
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(
            f"STEP 5/7 — Загрузка "
            f"{SLOW_DAYS}d / {SLOW_TIMEFRAME}"
        )
        print("=" * 70)

        slow_legs = []

        for candidate in fast_passed:
            slow_legs.append(
                candidate["leg_a"]
            )

            slow_legs.append(
                candidate["leg_b"]
            )

        slow_cache, slow_download_errors = (
            await load_many_ohlcv(
                exchanges=exchanges,
                legs=slow_legs,
                timeframe=SLOW_TIMEFRAME,
                days=SLOW_DAYS,
                progress_name=(
                    f"OHLCV {SLOW_TIMEFRAME}"
                ),
            )
        )

        print(
            f"Загружено unique ног: "
            f"{len(slow_cache)}"
        )

        print(
            f"Ошибок загрузки: "
            f"{len(slow_download_errors)}"
        )

        # ----------------------------------------------------
        # STEP 6 — SLOW ANALYSIS
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(
            f"STEP 6/7 — Анализ "
            f"{SLOW_DAYS}d / {SLOW_TIMEFRAME}"
        )
        print("=" * 70)

        slow_rows = []
        slow_passed = []

        for candidate in tqdm(
            fast_passed,
            desc="SLOW historical analysis",
        ):
            df_a = slow_cache.get(
                leg_key(candidate["leg_a"]),
                pd.DataFrame(
                    columns=["close", "volume"]
                ),
            )

            df_b = slow_cache.get(
                leg_key(candidate["leg_b"]),
                pd.DataFrame(
                    columns=["close", "volume"]
                ),
            )

            metrics = analyse_pair_ohlcv(
                df_a=df_a,
                df_b=df_b,
                timeframe=SLOW_TIMEFRAME,
            )

            row = candidate_row(candidate)

            row.update(
                metrics_to_row(
                    "fast",
                    candidate["fast_metrics"],
                )
            )

            row.update(
                metrics_to_row(
                    "slow",
                    metrics,
                )
            )

            slow_rows.append(row)

            if metrics.get(
                "historical_ok",
                False,
            ):
                candidate["slow_metrics"] = metrics

                slow_passed.append(
                    candidate
                )

        slow_df = pd.DataFrame(slow_rows)

        slow_path = os.path.join(
            OUTPUT_DIR,
            f"02_slow_all_{timestamp}.csv",
        )

        slow_df.to_csv(
            slow_path,
            index=False,
            encoding="utf-8-sig",
        )

        print()
        print(
            f"Прошли slow historical filter: "
            f"{len(slow_passed)} / {len(fast_passed)}"
        )

        if not slow_passed:
            print()
            print(
                "Ни одна связка не подтвердилась "
                "на 7d / 15m."
            )

            return {
                "fast_df": fast_df,
                "slow_df": slow_df,
                "tradeable_df": pd.DataFrame(),
                "execution_rejected_df": pd.DataFrame(),
            }

        # ----------------------------------------------------
        # STEP 7 — ORDERBOOK, VWAP, FUNDING
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(
            "STEP 7/7 — Стакан, VWAP, "
            "slippage, funding"
        )
        print("=" * 70)

        final_legs = {}

        for candidate in slow_passed:
            final_legs[
                leg_key(candidate["leg_a"])
            ] = candidate["leg_a"]

            final_legs[
                leg_key(candidate["leg_b"])
            ] = candidate["leg_b"]

        async def load_execution_data(
            key,
            leg,
        ):
            exchange = exchanges[
                leg["exchange_id"]
            ]

            book_task = fetch_orderbook_safe(
                exchange,
                leg["symbol"],
            )

            funding_task = fetch_funding_safe(
                exchange,
                leg["symbol"],
                leg["kind"],
            )

            orderbook, funding = await asyncio.gather(
                book_task,
                funding_task,
            )

            return key, orderbook, funding

        execution_tasks = [
            asyncio.create_task(
                load_execution_data(
                    key,
                    leg,
                )
            )
            for key, leg in final_legs.items()
        ]

        orderbooks = {}
        fundings = {}

        for future in tqdm(
            asyncio.as_completed(
                execution_tasks
            ),
            total=len(execution_tasks),
            desc="Стаканы и funding",
        ):
            key, orderbook, funding = await future

            orderbooks[key] = orderbook
            fundings[key] = funding

        tradeable_rows = []
        rejected_rows = []

        for candidate in tqdm(
            slow_passed,
            desc="Execution-фильтр",
        ):
            leg_a = candidate["leg_a"]
            leg_b = candidate["leg_b"]

            key_a = leg_key(leg_a)
            key_b = leg_key(leg_b)

            book_a = analyse_orderbook(
                orderbooks.get(key_a),
                TEST_NOTIONAL_USDT,
            )

            book_b = analyse_orderbook(
                orderbooks.get(key_b),
                TEST_NOTIONAL_USDT,
            )

            row = candidate_row(candidate)

            row.update(
                metrics_to_row(
                    "fast",
                    candidate["fast_metrics"],
                )
            )

            row.update(
                metrics_to_row(
                    "slow",
                    candidate["slow_metrics"],
                )
            )

            # ------------------------------------------------
            # Если стакан одной ноги не прошёл.
            # ------------------------------------------------

            if not book_a.get("ok", False):
                row.update({
                    "execution_ok": False,
                    "execution_rejection_reason": (
                        f"Нога A: "
                        f"{book_a.get('reason', 'unknown')}"
                    ),
                })

                rejected_rows.append(row)
                continue

            if not book_b.get("ok", False):
                row.update({
                    "execution_ok": False,
                    "execution_rejection_reason": (
                        f"Нога B: "
                        f"{book_b.get('reason', 'unknown')}"
                    ),
                })

                rejected_rows.append(row)
                continue

            # ------------------------------------------------
            # Реальные execution metrics.
            # ------------------------------------------------

            edges = executable_edges(
                book_a,
                book_b,
            )

            funding_a = fundings.get(
                key_a,
                {
                    "available": False,
                    "funding_rate": np.nan,
                },
            )

            funding_b = fundings.get(
                key_b,
                {
                    "available": False,
                    "funding_rate": np.nan,
                },
            )

            slow_metrics = candidate[
                "slow_metrics"
            ]

            half_life_days = (
                slow_metrics["half_life_bars"]
                * timeframe_to_minutes(
                    SLOW_TIMEFRAME
                )
                / 1_440
            )

            funding_a_rate = funding_a[
                "funding_rate"
            ]

            funding_b_rate = funding_b[
                "funding_rate"
            ]

            funding_known = (
                funding_a["available"]
                and funding_b["available"]
                and np.isfinite(funding_a_rate)
                and np.isfinite(funding_b_rate)
            )

            # Worst-case funding estimate.
            #
            # Для spot ставка = 0.
            # Для swap берём abs(rate), как adverse risk.
            if funding_known:
                adverse_funding_cost = (
                    abs(funding_a_rate)
                    + abs(funding_b_rate)
                ) * FUNDING_EVENTS_PER_DAY * max(
                    half_life_days,
                    1 / FUNDING_EVENTS_PER_DAY,
                )

            else:
                adverse_funding_cost = np.nan

            # Вход двумя ногами + выход двумя ногами.
            round_trip_fee_cost = (
                4 * DEFAULT_TAKER_FEE
            )

            # Slippage buy/sell на A и B,
            # умножаем на два для entry + exit.
            round_trip_slippage_cost = 2 * (
                book_a["buy_slippage"]
                + book_a["sell_slippage"]
                + book_b["buy_slippage"]
                + book_b["sell_slippage"]
            )

            direct_cost = (
                round_trip_fee_cost
                + round_trip_slippage_cost
            )

            total_cost = (
                direct_cost
                + adverse_funding_cost
                if funding_known
                else direct_cost
            )

            expected_one_sigma_move = (
                np.exp(
                    slow_metrics["spread_std"]
                ) - 1
            )

            max_leg_slippage = max(
                book_a["max_slippage"],
                book_b["max_slippage"],
            )

            liquidity_ok = (
                max_leg_slippage
                <= MAX_LEG_SLIPPAGE
            )

            move_covers_cost = (
                expected_one_sigma_move
                >= total_cost
                * MIN_COST_COVERAGE_MULTIPLIER
            )

            z = slow_metrics["latest_zscore"]

            if z >= 2:
                current_signal = (
                    "SHORT A / LONG B"
                )

            elif z <= -2:
                current_signal = (
                    "LONG A / SHORT B"
                )

            else:
                current_signal = (
                    "NO ENTRY: |Z| < 2"
                )

            row.update({
                "test_notional_usdt": (
                    TEST_NOTIONAL_USDT
                ),

                "a_best_bid": book_a["best_bid"],
                "a_best_ask": book_a["best_ask"],
                "a_buy_vwap": book_a["buy_vwap"],
                "a_sell_vwap": book_a["sell_vwap"],

                "a_buy_slippage_pct": (
                    book_a["buy_slippage"] * 100
                ),

                "a_sell_slippage_pct": (
                    book_a["sell_slippage"] * 100
                ),

                "b_best_bid": book_b["best_bid"],
                "b_best_ask": book_b["best_ask"],
                "b_buy_vwap": book_b["buy_vwap"],
                "b_sell_vwap": book_b["sell_vwap"],

                "b_buy_slippage_pct": (
                    book_b["buy_slippage"] * 100
                ),

                "b_sell_slippage_pct": (
                    book_b["sell_slippage"] * 100
                ),

                "max_leg_slippage_pct": (
                    max_leg_slippage * 100
                ),

                "edge_short_a_long_b_pct": (
                    edges["edge_short_a_long_b"]
                    * 100
                ),

                "edge_long_a_short_b_pct": (
                    edges["edge_long_a_short_b"]
                    * 100
                ),

                "best_current_executable_edge_pct": (
                    edges["best_current_edge"]
                    * 100
                ),

                "funding_known": funding_known,
                "funding_a": funding_a_rate,
                "funding_b": funding_b_rate,

                "estimated_adverse_funding_pct": (
                    adverse_funding_cost * 100
                    if np.isfinite(
                        adverse_funding_cost
                    )
                    else np.nan
                ),

                "round_trip_fee_cost_pct": (
                    round_trip_fee_cost * 100
                ),

                "round_trip_slippage_cost_pct": (
                    round_trip_slippage_cost * 100
                ),

                "estimated_total_cost_pct": (
                    total_cost * 100
                ),

                "expected_one_sigma_move_pct": (
                    expected_one_sigma_move * 100
                ),

                "current_signal": current_signal,
            })

            rejection_reasons = []

            if not liquidity_ok:
                rejection_reasons.append(
                    f"max slippage="
                    f"{max_leg_slippage * 100:.3f}% "
                    f"> limit="
                    f"{MAX_LEG_SLIPPAGE * 100:.3f}%"
                )

            if not move_covers_cost:
                rejection_reasons.append(
                    f"1σ move="
                    f"{expected_one_sigma_move * 100:.3f}% "
                    f"не покрывает costs="
                    f"{total_cost * 100:.3f}% "
                    f"× "
                    f"{MIN_COST_COVERAGE_MULTIPLIER:.2f}"
                )

            execution_ok = all([
                liquidity_ok,
                move_covers_cost,
            ])

            row["execution_ok"] = execution_ok

            row["execution_rejection_reason"] = (
                " | ".join(
                    rejection_reasons
                )
            )

            # Score для сортировки финалистов.
            adf_strength = -np.log10(
                max(
                    slow_metrics["adf_pvalue"],
                    1e-12,
                )
            )

            row["quality_score"] = (
                adf_strength
                + slow_metrics[
                    "zero_crossings_per_day"
                ]
                + slow_metrics[
                    "one_sigma_entries_per_day"
                ]
                + slow_metrics[
                    "q95_q05_range"
                ] * 20
                - total_cost * 100
            )

            if execution_ok:
                tradeable_rows.append(row)

            else:
                rejected_rows.append(row)

        tradeable_df = pd.DataFrame(
            tradeable_rows
        )

        execution_rejected_df = pd.DataFrame(
            rejected_rows
        )

        if not tradeable_df.empty:
            tradeable_df = (
                tradeable_df
                .sort_values(
                    [
                        "quality_score",
                        "slow_adf_pvalue",
                        "slow_zero_crossings_day",
                    ],
                    ascending=[
                        False,
                        True,
                        False,
                    ],
                )
                .reset_index(drop=True)
            )

        if not execution_rejected_df.empty:
            execution_rejected_df = (
                execution_rejected_df
                .sort_values(
                    [
                        "slow_adf_pvalue",
                        "slow_q95_q05_range_pct",
                    ],
                    ascending=[
                        True,
                        False,
                    ],
                )
                .reset_index(drop=True)
            )

        # ----------------------------------------------------
        # CSV OUTPUT
        # ----------------------------------------------------

        tradeable_path = os.path.join(
            OUTPUT_DIR,
            f"03_tradeable_{timestamp}.csv",
        )

        rejected_path = os.path.join(
            OUTPUT_DIR,
            f"04_execution_rejected_{timestamp}.csv",
        )

        tradeable_df.to_csv(
            tradeable_path,
            index=False,
            encoding="utf-8-sig",
        )

        execution_rejected_df.to_csv(
            rejected_path,
            index=False,
            encoding="utf-8-sig",
        )

        elapsed_seconds = (
            datetime.now(timezone.utc)
            - started_at
        ).total_seconds()

        print()
        print("=" * 70)
        print("СКАНИРОВАНИЕ ЗАВЕРШЕНО")
        print("=" * 70)

        print(
            f"Время сканирования: "
            f"{elapsed_seconds:.1f} сек"
        )

        print(
            f"Все кандидаты fast: "
            f"{len(fast_df)}"
        )

        print(
            f"Прошли fast: "
            f"{len(fast_passed)}"
        )

        print(
            f"Прошли slow: "
            f"{len(slow_passed)}"
        )

        print(
            f"Реально пригодны сейчас: "
            f"{len(tradeable_df)}"
        )

        print(
            f"Исторически подходят, "
            f"но execution reject: "
            f"{len(execution_rejected_df)}"
        )

        print()
        print("CSV сохранены:")
        print(fast_path)
        print(slow_path)
        print(tradeable_path)
        print(rejected_path)

        return {
            "fast_df": fast_df,
            "slow_df": slow_df,
            "tradeable_df": tradeable_df,
            "execution_rejected_df": (
                execution_rejected_df
            ),
        }

    finally:
        if exchanges:
            await asyncio.gather(
                *[
                    exchange.close()
                    for exchange in exchanges.values()
                ],
                return_exceptions=True,
            )


# ============================================================
# 8. ЗАПУСК СКАНЕРА
# ============================================================

results = await run_full_scanner()

fast_df = results["fast_df"]
slow_df = results["slow_df"]
tradeable_df = results["tradeable_df"]
execution_rejected_df = results[
    "execution_rejected_df"
]


# ============================================================
# 9. КОРОТКИЙ И ПОНЯТНЫЙ ВЫВОД
# ============================================================

print()
print("=" * 95)
print("ИТОГОВАЯ ТАБЛИЦА: РЕАЛЬНО ТОРГУЕМЫЕ СВЯЗКИ")
print("=" * 95)

if tradeable_df.empty:
    print(
        "Сейчас нет связок, прошедших "
        "и historical-фильтр, и execution-фильтр."
    )

    print()
    print(
        "Проверь execution_rejected_df: "
        "там связки с хорошей статистикой, "
        "которые отсеялись по стакану или costs."
    )

else:
    tradeable_display_columns = [
        "base",

        "exchange_a",
        "market_a",
        "symbol_a",

        "exchange_b",
        "market_b",
        "symbol_b",

        "slow_beta",
        "slow_adf_pvalue",

        "slow_q95_q05_range_pct",
        "slow_spread_std_pct",

        "slow_half_life_bars",
        "slow_zero_crossings_day",
        "slow_one_sigma_entries_day",

        "slow_latest_zscore",
        "current_signal",

        "max_leg_slippage_pct",

        "estimated_total_cost_pct",
        "expected_one_sigma_move_pct",

        "quality_score",
    ]

    tradeable_display_columns = [
        col
        for col in tradeable_display_columns
        if col in tradeable_df.columns
    ]

    display(
        tradeable_df[
            tradeable_display_columns
        ].head(50)
    )


print()
print("=" * 95)
print("ИСТОРИЧЕСКИ ХОРОШИЕ, НО НЕТОРГУЕМЫЕ СЕЙЧАС")
print("=" * 95)

if execution_rejected_df.empty:
    print(
        "Нет execution reject."
    )

else:
    rejected_display_columns = [
        "base",

        "exchange_a",
        "market_a",
        "symbol_a",

        "exchange_b",
        "market_b",
        "symbol_b",

        "slow_beta",
        "slow_adf_pvalue",

        "slow_q95_q05_range_pct",
        "slow_half_life_bars",

        "slow_zero_crossings_day",
        "slow_one_sigma_entries_day",

        "slow_latest_zscore",

        "test_notional_usdt",

        "max_leg_slippage_pct",

        "edge_short_a_long_b_pct",
        "edge_long_a_short_b_pct",

        "round_trip_fee_cost_pct",
        "round_trip_slippage_cost_pct",

        "estimated_adverse_funding_pct",
        "estimated_total_cost_pct",

        "expected_one_sigma_move_pct",

        "current_signal",
        "execution_rejection_reason",
    ]

    rejected_display_columns = [
        col
        for col in rejected_display_columns
        if col in execution_rejected_df.columns
    ]

    display(
        execution_rejected_df[
            rejected_display_columns
        ].head(50)
    )


print()
print("=" * 95)
print("TOP-20: САМЫЕ ШИРОКИЕ FAST SPREAD")
print("=" * 95)

if not fast_df.empty:
    fast_view = fast_df.copy()

    fast_view["pair_label"] = fast_view.apply(
        pair_label,
        axis=1,
    )

    fast_columns = [
        "pair_label",

        "fast_adf_pvalue",
        "fast_q95_q05_range_pct",
        "fast_spread_std_pct",

        "fast_half_life_bars",
        "fast_zero_crossings_day",
        "fast_one_sigma_entries_day",

        "fast_latest_zscore",

        "fast_historical_ok",
        "fast_historical_rejection_reason",
    ]

    fast_columns = [
        col
        for col in fast_columns
        if col in fast_view.columns
    ]

    display(
        fast_view
        .sort_values(
            "fast_q95_q05_range_pct",
            ascending=False,
        )[
            fast_columns
        ]
        .head(20)
    )


print()
print("=" * 95)
print("КАК ЧИТАТЬ ГЛАВНЫЕ ПОЛЯ")
print("=" * 95)

legend_df = pd.DataFrame(
    [
        {
            "Поле": "beta",
            "Значение": (
                "Коэффициент hedge ratio. "
                "Для одной монеты на разных биржах обычно близок к 1."
            ),
        },
        {
            "Поле": "ADF p-value",
            "Значение": (
                "Проверка стационарности spread. "
                "Чем ниже, тем лучше. Требование: ≤ 0.05."
            ),
        },
        {
            "Поле": "q95-q05 range %",
            "Значение": (
                "Устойчивый historical размах spread. "
                "Не единичный wick, а диапазон между 5-м и 95-м процентилем."
            ),
        },
        {
            "Поле": "half-life",
            "Значение": (
                "Оценка времени возврата spread к среднему, "
                "выраженная в барах."
            ),
        },
        {
            "Поле": "zero-crossings/day",
            "Значение": (
                "Сколько раз в день spread пересекает среднее. "
                "Это показатель частоты завершённых mean-reversion циклов."
            ),
        },
        {
            "Поле": "±1σ entries/day",
            "Значение": (
                "Как часто появляются умеренные отклонения, "
                "которые теоретически могут дать торговый вход."
            ),
        },
        {
            "Поле": "latest z-score",
            "Значение": (
                "Текущее отклонение spread от среднего. "
                "|Z| < 1 обычно не вход; около 2 — зона наблюдения/входа."
            ),
        },
        {
            "Поле": "max leg slippage %",
            "Значение": (
                "Реальная потеря на исполнении через стакан "
                "для TEST_NOTIONAL_USDT."
            ),
        },
        {
            "Поле": "estimated total cost %",
            "Значение": (
                "Комиссии + ожидаемое round-trip проскальзывание "
                "+ worst-case оценка funding."
            ),
        },
        {
            "Поле": "expected 1σ move %",
            "Значение": (
                "Обычный ожидаемый ход spread. "
                "Он должен заметно превышать total cost."
            ),
        },
    ]
)

display(legend_df)

STEP 1/7 — Подключение к биржам


mexc requires to release all resources with an explicit call to the .close() coroutine. If you are using the exchange instance with async coroutines, add `await exchange.close()` to your code into a place when you're done with the exchange and don't need the exchange instance anymore (at the end of your async coroutine).
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x00000237C1533C50>
gate requires to release all resources with an explicit call to the .close() coroutine. If you are using the exchange instance with async coroutines, add `await exchange.close()` to your code into a place when you're done with the exchange and don't need the exchange instance anymore (at the end of your async coroutine).
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x00000237ABD03250>


OK: mexc | markets=3109
OK: gate | markets=6827

STEP 2/7 — Spot / USDT-linear perpetual universe
mexc: spot=1424, linear swaps=1100
gate: spot=2053, linear swaps=1025

Всего связок: 562

STEP 3/7 — Загрузка 2d / 5m


OHLCV 5m:   0%|          | 0/1124 [00:00<?, ?it/s]

Загружено unique ног: 1124
Ошибок загрузки: 0

STEP 4/7 — Анализ 2d / 5m


...

,base,exchange_a,market_a,symbol_a,exchange_b,market_b,symbol_b,slow_beta,slow_adf_pvalue,slow_q95_q05_range_pct,slow_half_life_bars,slow_zero_crossings_day,slow_one_sigma_entries_day,slow_latest_zscore,test_notional_usdt,max_leg_slippage_pct,edge_short_a_long_b_pct,edge_long_a_short_b_pct,round_trip_fee_cost_pct,round_trip_slippage_cost_pct,estimated_adverse_funding_pct,estimated_total_cost_pct,expected_one_sigma_move_pct,current_signal,execution_rejection_reason
0,JUGGERNAUT,mexc,spot,JUGGERNAUT/USDT,gate,swap,JUGGERNAUT/USDT:USDT,0.934164,0.000000,6.738608,2.634448,19.200000,9.024000,-1.910274,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Нога B: Недостаточная bid-глубина для $100
1,CTK,mexc,spot,CTK/USDT,gate,swap,CTK/USDT:USDT,1.001621,0.000000,2.138027,2.638821,18.624000,9.600000,-2.647962,100.000000,0.227271,-1.499343,-0.430912,0.400000,1.197721,0.122400,1.720121,0.632742,LONG A / SHORT B,1σ move=0.633% не покрывает costs=1.720% × 1.50
2,SIREN,mexc,spot,SIREN/USDT,gate,swap,SIREN/USDT:USDT,0.379853,0.000000,3.153328,2.788267,17.472000,6.720000,-1.171188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Нога B: Недостаточная ask-глубина для $100
3,INDEX,mexc,spot,INDEX/USDT,gate,swap,INDEX/USDT:USDT,0.833481,0.000000,6.821331,2.400360,18.816000,10.752000,-0.560251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Нога B: Недостаточная ask-глубина для $100
4,BP,mexc,spot,BP/USDT,gate,swap,BP/USDT:USDT,1.028792,0.001559,5.348036,3.891902,15.360000,10.176000,1.061221,100.000000,1.540839,-10.839938,6.733058,0.400000,4.981186,0.005000,5.386186,1.790660,NO ENTRY: |Z| < 2,max slippage=1.541% > limit=0.450% | 1σ move=1...



TOP-20: САМЫЕ ШИРОКИЕ FAST SPREAD


,pair_label,fast_adf_pvalue,fast_q95_q05_range_pct,fast_spread_std_pct,fast_half_life_bars,fast_zero_crossings_day,fast_one_sigma_entries_day,fast_latest_zscore,fast_historical_ok,fast_historical_rejection_reason
188,EWT | MEXC SPOT (EWT/USDT) ↔ GATE SWAP (EWT/US...,0.239572,21.450889,7.225471,58.439204,6.336000,2.880000,0.205402,False,ADF=0.2396 > 0.05
392,PURR | MEXC SPOT (PURR/USDT) ↔ GATE SWAP (PURR...,0.203819,13.672100,4.421271,39.646981,20.160000,4.608000,2.515765,False,ADF=0.2038 > 0.05
166,EACC | MEXC SPOT (EACC/USDT) ↔ GATE SWAP (EACC...,0.001087,12.741878,4.095942,5.941920,39.168000,23.040000,-0.600721,True,
407,RIO | MEXC SPOT (RIO/USDT) ↔ GATE SWAP (RIO/US...,0.004583,10.162833,2.803735,10.050035,24.768000,12.096000,0.217628,True,
520,WEN | MEXC SPOT (WEN/USDT) ↔ GATE SWAP (WEN/US...,0.040891,8.539202,2.660369,13.976551,15.552000,17.856000,-0.139250,True,
545,ZCAT | MEXC SPOT (ZCAT/USDT) ↔ GATE SWAP (ZCAT...,0.003971,8.347548,2.619180,2.889855,48.960000,43.200000,-1.188907,True,
204,FLYBRAIN | MEXC SPOT (FLYBRAIN/USDT) ↔ GATE SW...,0.000442,8.170146,2.549824,1.657496,81.792000,49.536000,-0.294087,False,half-life=1.7 bars
370,PEP | MEXC SPOT (PEP/USDT) ↔ GATE SWAP (PEP/US...,0.085784,6.411456,2.100244,9.101759,11.520000,9.792000,-0.807127,False,ADF=0.0858 > 0.05
217,GME | MEXC SPOT (GME/USDT) ↔ GATE SWAP (GME/US...,0.497998,6.228140,2.092061,37.381178,6.912000,13.248000,1.144211,False,ADF=0.4980 > 0.05
221,GP | MEXC SPOT (GP/USDT) ↔ GATE SWAP (GP/USDT:...,0.000005,6.225351,1.823428,1.672825,80.064000,36.288000,2.156515,False,half-life=1.7 bars



КАК ЧИТАТЬ ГЛАВНЫЕ ПОЛЯ


,Поле,Значение
0,beta,Коэффициент hedge ratio. Для одной монеты на р...
1,ADF p-value,"Проверка стационарности spread. Чем ниже, тем ..."
2,q95-q05 range %,Устойчивый historical размах spread. Не единич...
3,half-life,"Оценка времени возврата spread к среднему, выр..."
4,zero-crossings/day,Сколько раз в день spread пересекает среднее. ...
5,±1σ entries/day,"Как часто появляются умеренные отклонения, кот..."
6,latest z-score,Текущее отклонение spread от среднего. |Z| < 1...
7,max leg slippage %,Реальная потеря на исполнении через стакан для...
8,estimated total cost %,Комиссии + ожидаемое round-trip проскальзывани...
9,expected 1σ move %,Обычный ожидаемый ход spread. Он должен заметн...


In [11]:
# ============================================================
# РАЗБОР РЕЗУЛЬТАТОВ СКАНЕРА
#
# Требует, чтобы уже существовали:
# fast_df
# slow_df
# tradeable_df
# execution_rejected_df
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


# ============================================================
# 1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def safe_col(df, column, default=np.nan):
    """
    Вернуть колонку, если она существует.
    Иначе вернуть Series из default.
    """
    if column in df.columns:
        return df[column]

    return pd.Series(
        [default] * len(df),
        index=df.index,
    )


def market_label(row):
    """
    Читаемая подпись связки.
    """
    return (
        f"{row['base']} | "
        f"{row['exchange_a'].upper()} {row['market_a'].upper()} "
        f"({row['symbol_a']})  ↔  "
        f"{row['exchange_b'].upper()} {row['market_b'].upper()} "
        f"({row['symbol_b']})"
    )


def pct(value, digits=3):
    """
    Удобное форматирование процентов.
    """
    if pd.isna(value):
        return "нет данных"

    return f"{value:.{digits}f}%"


def num(value, digits=3):
    """
    Удобное форматирование числа.
    """
    if pd.isna(value):
        return "нет данных"

    if np.isinf(value):
        return "∞"

    return f"{value:.{digits}f}"


def get_fast_failure_flags(df):
    """
    Причины, по которым пары не прошли FAST 2d / 5m этап.
    """
    flags = pd.DataFrame(index=df.index)

    flags["Не прошла стационарность ADF"] = (
        safe_col(df, "fast_stationary", False) == False
    )

    flags["Размах меньше заданного"] = (
        safe_col(df, "fast_range_ok", False) == False
    )

    flags["Слишком маленькое std"] = (
        safe_col(df, "fast_std_ok", False) == False
    )

    flags["Плохой half-life"] = (
        safe_col(df, "fast_half_life_ok", False) == False
    )

    flags["Редко пересекает среднее"] = (
        safe_col(df, "fast_crossings_ok", False) == False
    )

    flags["Редко входит в ±1σ"] = (
        safe_col(df, "fast_activity_ok", False) == False
    )

    return flags


def get_slow_failure_flags(df):
    """
    Причины, по которым FAST-финалисты не прошли SLOW 7d / 15m этап.
    """
    flags = pd.DataFrame(index=df.index)

    flags["Не прошла стационарность ADF"] = (
        safe_col(df, "slow_stationary", False) == False
    )

    flags["Размах меньше заданного"] = (
        safe_col(df, "slow_range_ok", False) == False
    )

    flags["Слишком маленькое std"] = (
        safe_col(df, "slow_std_ok", False) == False
    )

    flags["Плохой half-life"] = (
        safe_col(df, "slow_half_life_ok", False) == False
    )

    flags["Редко пересекает среднее"] = (
        safe_col(df, "slow_crossings_ok", False) == False
    )

    flags["Редко входит в ±1σ"] = (
        safe_col(df, "slow_activity_ok", False) == False
    )

    return flags


# ============================================================
# 2. ОБЩАЯ СВОДКА
# ============================================================

total_fast = len(fast_df)
passed_fast = int(
    safe_col(fast_df, "fast_historical_ok", False).sum()
)

total_slow = len(slow_df)
passed_slow = int(
    safe_col(slow_df, "slow_historical_ok", False).sum()
)

tradeable_count = len(tradeable_df)
execution_rejected_count = len(execution_rejected_df)

summary_df = pd.DataFrame(
    [
        {
            "Этап": "Все сформированные связки",
            "Количество": total_fast,
            "Что это означает": "Одинаковый underlying asset на Binance и Bybit, Spot/Swap согласно настройкам",
        },
        {
            "Этап": "Прошли 2 дня / 5 минут",
            "Количество": passed_fast,
            "Что это означает": "Прошли ADF, размах, std, half-life и частоту разворотов на быстром окне",
        },
        {
            "Этап": "Прошли 7 дней / 15 минут",
            "Количество": passed_slow,
            "Что это означает": "Подтвердили те же свойства на более длинном окне",
        },
        {
            "Этап": "Реально исполнимы сейчас",
            "Количество": tradeable_count,
            "Что это означает": "Стакан, slippage и оценка затрат не уничтожают потенциальное движение",
        },
        {
            "Этап": "Исторически подходят, но execution reject",
            "Количество": execution_rejected_count,
            "Что это означает": "График хороший, но текущая сделка не проходит фильтр ликвидности или costs",
        },
    ]
)

print("=" * 95)
print("ОБЩАЯ СВОДКА СКАНИРОВАНИЯ")
print("=" * 95)

display(summary_df)


# ============================================================
# 3. ЧТО УБИЛО БОЛЬШИНСТВО ПАР НА FAST ЭТАПЕ
# ============================================================

print()
print("=" * 95)
print("FAST 2D / 5M — КАКИЕ ФИЛЬТРЫ ЧАЩЕ ВСЕГО ОТСЕКАЮТ ПАРЫ")
print("=" * 95)

fast_flags = get_fast_failure_flags(fast_df)

fast_failure_summary = pd.DataFrame({
    "Фильтр": fast_flags.columns,
    "Отсеяно пар": [
        int(fast_flags[column].sum())
        for column in fast_flags.columns
    ],
})

fast_failure_summary["Доля от всех кандидатов, %"] = (
    fast_failure_summary["Отсеяно пар"]
    / max(total_fast, 1)
    * 100
)

fast_failure_summary = fast_failure_summary.sort_values(
    "Отсеяно пар",
    ascending=False,
).reset_index(drop=True)

display(fast_failure_summary)

top_fast_failure = fast_failure_summary.iloc[0]

print(
    f"Главный ограничитель сейчас: "
    f"«{top_fast_failure['Фильтр']}» — "
    f"{int(top_fast_failure['Отсеяно пар'])} из {total_fast} пар "
    f"({top_fast_failure['Доля от всех кандидатов, %']:.1f}%)."
)


# ============================================================
# 4. ЛУЧШИЕ ПАРЫ, НЕ ПРОШЕДШИЕ FAST
# ============================================================

print()
print("=" * 95)
print("TOP-30 БЛИЖАЙШИХ К ПРОХОЖДЕНИЮ FAST-ФИЛЬТРА")
print("=" * 95)

fast_near_miss = fast_df.copy()

fast_near_miss["pair_label"] = fast_near_miss.apply(
    market_label,
    axis=1,
)

# Чем ближе к 0, тем лучше ADF.
fast_near_miss["adf_distance"] = (
    safe_col(fast_near_miss, "fast_adf_pvalue", 1.0) - ADF_PVALUE_MAX
).clip(lower=0)

# Насколько процентов не хватает до заданного range.
fast_near_miss["range_missing_pct"] = (
    MIN_Q95_Q05_RANGE * 100
    - safe_col(
        fast_near_miss,
        "fast_q95_q05_range_pct",
        0.0,
    )
).clip(lower=0)

# Насколько недостаёт zero crossing.
fast_near_miss["crossings_missing"] = (
    MIN_ZERO_CROSSINGS_PER_DAY
    - safe_col(
        fast_near_miss,
        "fast_zero_crossings_day",
        0.0,
    )
).clip(lower=0)

# Насколько недостаёт входов в ±1 sigma.
fast_near_miss["activity_missing"] = (
    MIN_ONE_SIGMA_ENTRIES_PER_DAY
    - safe_col(
        fast_near_miss,
        "fast_one_sigma_entries_day",
        0.0,
    )
).clip(lower=0)

# Чем меньше score, тем ближе пара к твоим правилам.
fast_near_miss["distance_score"] = (
    fast_near_miss["adf_distance"] * 30
    + fast_near_miss["range_missing_pct"] / 4
    + fast_near_miss["crossings_missing"] / 3
    + fast_near_miss["activity_missing"] / 3
)

fast_near_miss = fast_near_miss.sort_values(
    [
        "distance_score",
        "fast_adf_pvalue",
        "fast_q95_q05_range_pct",
    ],
    ascending=[
        True,
        True,
        False,
    ],
)

fast_near_miss_columns = [
    "pair_label",
    "fast_adf_pvalue",
    "fast_q95_q05_range_pct",
    "fast_spread_std_pct",
    "fast_half_life_bars",
    "fast_zero_crossings_day",
    "fast_one_sigma_entries_day",
    "fast_latest_zscore",
    "fast_historical_ok",
    "fast_historical_rejection_reason",
]

fast_near_miss_columns = [
    column
    for column in fast_near_miss_columns
    if column in fast_near_miss.columns
]

display(
    fast_near_miss[
        fast_near_miss_columns
    ].head(30)
)


# ============================================================
# 5. ПАРА, КОТОРАЯ ПРОШЛА ИСТОРИЧЕСКИЕ УСЛОВИЯ
# ============================================================

print()
print("=" * 95)
print("ИСТОРИЧЕСКИ ПРОШЕДШИЕ СВЯЗКИ")
print("=" * 95)

if slow_df.empty:
    print(
        "Ни одна связка не прошла fast этап, "
        "поэтому slow-анализ отсутствует."
    )

else:
    slow_historical_passed = slow_df[
        safe_col(slow_df, "slow_historical_ok", False) == True
    ].copy()

    if slow_historical_passed.empty:
        print(
            "Fast-финалисты не подтвердили условия "
            "на 7d / 15m."
        )

        slow_fail_columns = [
            "base",
            "exchange_a",
            "market_a",
            "exchange_b",
            "market_b",
            "slow_adf_pvalue",
            "slow_q95_q05_range_pct",
            "slow_half_life_bars",
            "slow_zero_crossings_day",
            "slow_one_sigma_entries_day",
            "slow_historical_rejection_reason",
        ]

        slow_fail_columns = [
            column
            for column in slow_fail_columns
            if column in slow_df.columns
        ]

        display(slow_df[slow_fail_columns])

    else:
        slow_historical_passed["pair_label"] = (
            slow_historical_passed.apply(
                market_label,
                axis=1,
            )
        )

        historical_columns = [
            "pair_label",

            "fast_beta",
            "fast_adf_pvalue",
            "fast_q95_q05_range_pct",
            "fast_half_life_bars",
            "fast_zero_crossings_day",
            "fast_one_sigma_entries_day",

            "slow_beta",
            "slow_adf_pvalue",
            "slow_q95_q05_range_pct",
            "slow_spread_std_pct",
            "slow_half_life_bars",
            "slow_zero_crossings_day",
            "slow_one_sigma_entries_day",
            "slow_two_sigma_entries_day",
            "slow_latest_zscore",
            "slow_latest_spread_pct",
        ]

        historical_columns = [
            column
            for column in historical_columns
            if column in slow_historical_passed.columns
        ]

        display(
            slow_historical_passed[
                historical_columns
            ]
        )

        for _, row in slow_historical_passed.iterrows():
            print()
            print("-" * 95)
            print(market_label(row))
            print("-" * 95)

            print(
                f"β hedge ratio: "
                f"{num(row.get('slow_beta'))}"
            )

            print(
                f"ADF p-value: "
                f"{num(row.get('slow_adf_pvalue'), 5)} "
                f"(требование ≤ {ADF_PVALUE_MAX})"
            )

            print(
                f"Устойчивый range q95–q05: "
                f"{pct(row.get('slow_q95_q05_range_pct'))}"
            )

            print(
                f"Std spread: "
                f"{pct(row.get('slow_spread_std_pct'))}"
            )

            print(
                f"Half-life: "
                f"{num(row.get('slow_half_life_bars'), 1)} bars "
                f"≈ {num(row.get('slow_half_life_bars', np.nan) * 15 / 60, 1)} hours"
            )

            print(
                f"Zero-crossings: "
                f"{num(row.get('slow_zero_crossings_day'), 2)} / day"
            )

            print(
                f"Заходы в ±1σ: "
                f"{num(row.get('slow_one_sigma_entries_day'), 2)} / day"
            )

            print(
                f"Текущий Z-score: "
                f"{num(row.get('slow_latest_zscore'), 3)}"
            )

            z = row.get("slow_latest_zscore", np.nan)

            if pd.notna(z):
                if z >= 2:
                    print(
                        "Сигнал: spread существенно выше среднего. "
                        "Статистическое направление: SHORT A / LONG B."
                    )
                elif z <= -2:
                    print(
                        "Сигнал: spread существенно ниже среднего. "
                        "Статистическое направление: LONG A / SHORT B."
                    )
                else:
                    print(
                        "Сигнал: сейчас нет экстремального Z-score. "
                        "Пара может быть пригодной структурно, "
                        "но точки входа прямо сейчас нет."
                    )


# ============================================================
# 6. ПОЧЕМУ ПРОВАЛИЛАСЬ РЕАЛЬНАЯ ИСПОЛНИМОСТЬ
# ============================================================

print()
print("=" * 95)
print("EXECUTION REJECT — ПОЧЕМУ ИСТОРИЧЕСКИ ХОРОШАЯ ПАРА НЕЛЬЗЯ ТОРГОВАТЬ")
print("=" * 95)

if execution_rejected_df.empty:
    print(
        "Нет execution reject. "
        "Либо tradeable_df содержит подходящие пары, "
        "либо до проверки стакана ни одна пара не дошла."
    )

else:
    execution_view = execution_rejected_df.copy()

    execution_view["pair_label"] = execution_view.apply(
        market_label,
        axis=1,
    )

    execution_columns = [
        "pair_label",

        "slow_beta",
        "slow_adf_pvalue",
        "slow_q95_q05_range_pct",
        "slow_half_life_bars",
        "slow_zero_crossings_day",
        "slow_one_sigma_entries_day",
        "slow_latest_zscore",

        "test_notional_usdt",

        "max_leg_slippage_pct",

        "edge_short_a_long_b_pct",
        "edge_long_a_short_b_pct",
        "best_current_executable_edge_pct",

        "round_trip_fee_cost_pct",
        "round_trip_slippage_cost_pct",
        "estimated_adverse_funding_pct",
        "estimated_total_cost_pct",

        "expected_one_sigma_move_pct",

        "funding_known",
        "funding_a",
        "funding_b",

        "current_signal",
        "execution_rejection_reason",
    ]

    execution_columns = [
        column
        for column in execution_columns
        if column in execution_view.columns
    ]

    display(
        execution_view[
            execution_columns
        ]
    )

    for _, row in execution_view.iterrows():
        print()
        print("-" * 95)
        print(market_label(row))
        print("-" * 95)

        print(
            f"Проверочный размер: "
            f"${row.get('test_notional_usdt', np.nan):,.0f} на каждую ногу"
        )

        print(
            f"Максимальное проскальзывание одной ноги: "
            f"{pct(row.get('max_leg_slippage_pct'))}"
        )

        print(
            f"Текущий executable edge "
            f"(short A / long B): "
            f"{pct(row.get('edge_short_a_long_b_pct'))}"
        )

        print(
            f"Текущий executable edge "
            f"(long A / short B): "
            f"{pct(row.get('edge_long_a_short_b_pct'))}"
        )

        print(
            f"Round-trip комиссии: "
            f"{pct(row.get('round_trip_fee_cost_pct'))}"
        )

        print(
            f"Round-trip slippage: "
            f"{pct(row.get('round_trip_slippage_cost_pct'))}"
        )

        print(
            f"Оценка adverse funding: "
            f"{pct(row.get('estimated_adverse_funding_pct'))}"
        )

        print(
            f"Оценка total cost: "
            f"{pct(row.get('estimated_total_cost_pct'))}"
        )

        print(
            f"Ожидаемое движение 1σ: "
            f"{pct(row.get('expected_one_sigma_move_pct'))}"
        )

        print()

        print(
            "ИТОГОВАЯ ПРИЧИНА ОТСЕВА: "
            f"{row.get('execution_rejection_reason', 'не указана')}"
        )


# ============================================================
# 7. ЛУЧШИЕ КАНДИДАТЫ ПО РАЗМАХУ SPREAD
# ============================================================

print()
print("=" * 95)
print("TOP-30 САМЫХ ШИРОКИХ FAST-SPREAD")
print("=" * 95)

widest_spreads = fast_df.copy()

widest_spreads["pair_label"] = widest_spreads.apply(
    market_label,
    axis=1,
)

widest_columns = [
    "pair_label",
    "fast_q95_q05_range_pct",
    "fast_spread_std_pct",
    "fast_adf_pvalue",
    "fast_half_life_bars",
    "fast_zero_crossings_day",
    "fast_one_sigma_entries_day",
    "fast_historical_ok",
    "fast_historical_rejection_reason",
]

widest_columns = [
    column
    for column in widest_columns
    if column in widest_spreads.columns
]

display(
    widest_spreads.sort_values(
        "fast_q95_q05_range_pct",
        ascending=False,
    )[widest_columns].head(30)
)


# ============================================================
# 8. ЛУЧШИЕ СТАЦИОНАРНЫЕ ПАРЫ, ДАЖЕ ЕСЛИ РАЗМАХ МАЛЕНЬКИЙ
# ============================================================

print()
print("=" * 95)
print("TOP-30 САМЫХ СТАЦИОНАРНЫХ FAST-SPREAD")
print("=" * 95)

stationary_view = fast_df.copy()

stationary_view["pair_label"] = stationary_view.apply(
    market_label,
    axis=1,
)

stationary_columns = [
    "pair_label",
    "fast_adf_pvalue",
    "fast_q95_q05_range_pct",
    "fast_spread_std_pct",
    "fast_half_life_bars",
    "fast_zero_crossings_day",
    "fast_one_sigma_entries_day",
    "fast_historical_ok",
    "fast_historical_rejection_reason",
]

stationary_columns = [
    column
    for column in stationary_columns
    if column in stationary_view.columns
]

display(
    stationary_view.sort_values(
        "fast_adf_pvalue",
        ascending=True,
    )[stationary_columns].head(30)
)


# ============================================================
# 9. КОРОТКАЯ ИНТЕРПРЕТАЦИЯ ОБОЗНАЧЕНИЙ
# ============================================================

print()
print("=" * 95)
print("КАК ЧИТАТЬ ОСНОВНЫЕ ПОЛЯ")
print("=" * 95)

legend_df = pd.DataFrame(
    [
        {
            "Поле": "β / beta",
            "Смысл": "Hedge ratio. Для одной монеты на разных биржах обычно около 1. Не является готовым размером ордера без учёта contract size.",
        },
        {
            "Поле": "ADF p-value",
            "Смысл": "Проверка стационарности. Меньше 0.05 — spread статистически ближе к mean-reverting режиму.",
        },
        {
            "Поле": "q95-q05 range %",
            "Смысл": "Устойчивый размах spread без единичных экстремальных wick. Это твой основной фильтр «достаточно широкого хода».",
        },
        {
            "Поле": "spread std %",
            "Смысл": "Обычная амплитуда колебаний. Нужна, чтобы движение было больше издержек, но не выглядело как сломанный рынок.",
        },
        {
            "Поле": "half-life",
            "Смысл": "Оценка времени, за которое spread проходит половину пути к среднему. Слишком большой — капитал долго висит; бесконечность — возврат к mean не доказан.",
        },
        {
            "Поле": "zero-crossings/day",
            "Смысл": "Сколько раз в сутки z-score проходит через ноль. Больше — больше завершённых циклов возврата к mean.",
        },
        {
            "Поле": "±1σ entries/day",
            "Смысл": "Сколько раз в сутки появляются умеренные отклонения, из которых теоретически можно строить сделки.",
        },
        {
            "Поле": "latest z-score",
            "Смысл": "Текущая удалённость spread от среднего. Обычно |Z| < 1 — нет входа; около 2 — зона наблюдения/входа после costs; > 3 — повышенный риск structural break.",
        },
        {
            "Поле": "max leg slippage %",
            "Смысл": "Потеря от исполнения по нескольким уровням стакана для TEST_NOTIONAL_USDT. Если выше лимита, spread может существовать только на экране.",
        },
        {
            "Поле": "estimated total cost %",
            "Смысл": "Консервативная оценка fees + entry/exit slippage + adverse funding. Это не финальный бэктест, а фильтр явной нереализуемости.",
        },
        {
            "Поле": "expected 1σ move %",
            "Смысл": "Обычный потенциальный ход spread. Он должен заметно перекрывать total cost, иначе статистически красивая связка не имеет экономического смысла.",
        },
    ]
)

display(legend_df)

ОБЩАЯ СВОДКА СКАНИРОВАНИЯ


,Этап,Количество,Что это означает
0,Все сформированные связки,562,Одинаковый underlying asset на Binance и Bybit...
1,Прошли 2 дня / 5 минут,27,"Прошли ADF, размах, std, half-life и частоту р..."
2,Прошли 7 дней / 15 минут,5,Подтвердили те же свойства на более длинном окне
3,Реально исполнимы сейчас,0,"Стакан, slippage и оценка затрат не уничтожают..."
4,"Исторически подходят, но execution reject",5,"График хороший, но текущая сделка не проходит ..."



FAST 2D / 5M — КАКИЕ ФИЛЬТРЫ ЧАЩЕ ВСЕГО ОТСЕКАЮТ ПАРЫ


,Фильтр,Отсеяно пар,"Доля от всех кандидатов, %"
0,Не прошла стационарность ADF,562,100.0000
1,Размах меньше заданного,562,100.0000
2,Слишком маленькое std,562,100.0000
3,Плохой half-life,562,100.0000
4,Редко пересекает среднее,562,100.0000
5,Редко входит в ±1σ,562,100.0000


Главный ограничитель сейчас: «Не прошла стационарность ADF» — 562 из 562 пар (100.0%).

TOP-30 БЛИЖАЙШИХ К ПРОХОЖДЕНИЮ FAST-ФИЛЬТРА


...

,pair_label,fast_q95_q05_range_pct,fast_spread_std_pct,fast_adf_pvalue,fast_half_life_bars,fast_zero_crossings_day,fast_one_sigma_entries_day,fast_historical_ok,fast_historical_rejection_reason
188,EWT | MEXC SPOT (EWT/USDT) ↔ GATE SWAP (EWT/...,21.4509,7.2255,0.2396,58.4392,6.3360,2.8800,False,ADF=0.2396 > 0.05
392,PURR | MEXC SPOT (PURR/USDT) ↔ GATE SWAP (PU...,13.6721,4.4213,0.2038,39.6470,20.1600,4.6080,False,ADF=0.2038 > 0.05
166,EACC | MEXC SPOT (EACC/USDT) ↔ GATE SWAP (EA...,12.7419,4.0959,0.0011,5.9419,39.1680,23.0400,True,
407,RIO | MEXC SPOT (RIO/USDT) ↔ GATE SWAP (RIO/...,10.1628,2.8037,0.0046,10.0500,24.7680,12.0960,True,
520,WEN | MEXC SPOT (WEN/USDT) ↔ GATE SWAP (WEN/...,8.5392,2.6604,0.0409,13.9766,15.5520,17.8560,True,
545,ZCAT | MEXC SPOT (ZCAT/USDT) ↔ GATE SWAP (ZC...,8.3475,2.6192,0.0040,2.8899,48.9600,43.2000,True,
204,FLYBRAIN | MEXC SPOT (FLYBRAIN/USDT) ↔ GATE ...,8.1701,2.5498,0.0004,1.6575,81.7920,49.5360,False,half-life=1.7 bars
370,PEP | MEXC SPOT (PEP/USDT) ↔ GATE SWAP (PEP/...,6.4115,2.1002,0.0858,9.1018,11.5200,9.7920,False,ADF=0.0858 > 0.05
217,GME | MEXC SPOT (GME/USDT) ↔ GATE SWAP (GME/...,6.2281,2.0921,0.4980,37.3812,6.9120,13.2480,False,ADF=0.4980 > 0.05
221,GP | MEXC SPOT (GP/USDT) ↔ GATE SWAP (GP/USD...,6.2254,1.8234,0.0000,1.6728,80.0640,36.2880,False,half-life=1.7 bars



TOP-30 САМЫХ СТАЦИОНАРНЫХ FAST-SPREAD


,pair_label,fast_adf_pvalue,fast_q95_q05_range_pct,fast_spread_std_pct,fast_half_life_bars,fast_zero_crossings_day,fast_one_sigma_entries_day,fast_historical_ok,fast_historical_rejection_reason
3,4 | MEXC SPOT (4/USDT) ↔ GATE SWAP (4/USDT:U...,0.0000,0.6459,0.1938,0.8009,133.0560,61.0560,False,range=0.65% вне 2.00%–35.00% | std=0.194% < 0....
24,AKE | MEXC SPOT (AKE/USDT) ↔ GATE SWAP (AKE/...,0.0000,0.4119,0.1260,0.7201,143.4240,69.6960,False,range=0.41% вне 2.00%–35.00% | std=0.126% < 0....
34,APE | MEXC SPOT (APE/USDT) ↔ GATE SWAP (APE/...,0.0000,0.3314,0.1058,0.8028,138.2400,56.4480,False,range=0.33% вне 2.00%–35.00% | std=0.106% < 0....
26,ALGO | MEXC SPOT (ALGO/USDT) ↔ GATE SWAP (AL...,0.0000,0.3302,0.1003,0.7348,138.2400,62.7840,False,range=0.33% вне 2.00%–35.00% | std=0.100% < 0....
515,VVV | MEXC SPOT (VVV/USDT) ↔ GATE SWAP (VVV/...,0.0000,0.3410,0.1204,0.7631,133.0560,57.0240,False,range=0.34% вне 2.00%–35.00% | std=0.120% < 0....
517,WAL | MEXC SPOT (WAL/USDT) ↔ GATE SWAP (WAL/...,0.0000,0.3490,0.1113,0.7881,145.7280,62.2080,False,range=0.35% вне 2.00%–35.00% | std=0.111% < 0....
37,APT | MEXC SPOT (APT/USDT) ↔ GATE SWAP (APT/...,0.0000,0.2265,0.0698,0.7532,134.7840,66.2400,False,range=0.23% вне 2.00%–35.00% | std=0.070% < 0....
39,ARB | MEXC SPOT (ARB/USDT) ↔ GATE SWAP (ARB/...,0.0000,0.2514,0.0758,0.7096,132.4800,66.8160,False,range=0.25% вне 2.00%–35.00% | std=0.076% < 0....
502,US | MEXC SPOT (US/USDT) ↔ GATE SWAP (US/USD...,0.0000,0.9373,0.3081,0.7487,137.6640,59.3280,False,range=0.94% вне 2.00%–35.00% | std=0.308% < 0....
55,AVAX | MEXC SPOT (AVAX/USDT) ↔ GATE SWAP (AV...,0.0000,0.2013,0.0628,0.7211,127.2960,54.1440,False,range=0.20% вне 2.00%–35.00% | std=0.063% < 0....



КАК ЧИТАТЬ ОСНОВНЫЕ ПОЛЯ


,Поле,Смысл
0,β / beta,Hedge ratio. Для одной монеты на разных биржах...
1,ADF p-value,Проверка стационарности. Меньше 0.05 — spread ...
2,q95-q05 range %,Устойчивый размах spread без единичных экстрем...
3,spread std %,"Обычная амплитуда колебаний. Нужна, чтобы движ..."
4,half-life,"Оценка времени, за которое spread проходит пол..."
5,zero-crossings/day,Сколько раз в сутки z-score проходит через нол...
6,±1σ entries/day,Сколько раз в сутки появляются умеренные откло...
7,latest z-score,Текущая удалённость spread от среднего. Обычно...
8,max leg slippage %,Потеря от исполнения по нескольким уровням ста...
9,estimated total cost %,Консервативная оценка fees + entry/exit slippa...
